# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# 0.1 Mount Google Drive để lưu checkpoint/adapter đề phòng mất session
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 0.2 Đồng bộ mã nguồn repo nếu chạy trên Colab
import os
REPO_DIR = "/content/K4-Track-4-Day10-VisionLanguageModel"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/awun0105/K4-Track-4-Day10-VisionLanguageModel.git /content/K4-Track-4-Day10-VisionLanguageModel
%cd /content/K4-Track-4-Day10-VisionLanguageModel

In [ ]:
# 0.3 Kiểm tra T4 GPU và cài đặt thư viện
# Lưu ý: Chọn menu Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# 0.4 Cấu hình thư mục lưu checkpoint trên Google Drive
import shutil
DRIVE_BACKUP_DIR = "/content/drive/MyDrive/AI20K_Lab_Day10_Checkpoints"
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

def backup_to_drive(src_path, exp_name):
    """Hàm tiện ích sao lưu checkpoint/adapter sang Google Drive"""
    dest = os.path.join(DRIVE_BACKUP_DIR, exp_name)
    if os.path.isdir(src_path):
        shutil.copytree(src_path, dest, dirs_exist_ok=True)
    elif os.path.isfile(src_path):
        shutil.copy2(src_path, dest)
    print(f"✅ Đã sao lưu {src_path} -> {dest}")

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# YOUR CODE HERE: đọc data/splits/train.jsonl, xem vài mẫu, vẽ box lên ảnh
import json
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

with open("data/splits/train.jsonl", "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f][:3]

for item in samples:
    img = Image.open(f"data/{item['image_path']}")
    draw = ImageDraw.Draw(img)
    w, h = item["width"], item["height"]
    for obj in item["objects"]:
        xc, yc, bw, bh = obj["yolo"]
        x1, y1 = (xc - bw/2) * w, (yc - bh/2) * h
        x2, y2 = (xc + bw/2) * w, (yc + bh/2) * h
        draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
    plt.figure(figsize=(6, 4))
    plt.imshow(img)
    plt.title(f"Image ID: {item['image_id']} | Objects: {len(item['objects'])}")
    plt.axis("off")
    plt.show()

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# YOUR CODE HERE: viết yolo_to_box() và objects_to_answer() (GUIDE bước 2)
import json


def yolo_to_box(label: str, yolo: list[float]) -> dict:
    """
    Chuyển đổi 1 nhãn YOLO [xc, yc, w, h] chuẩn hóa 0..1
    sang box Qwen Grounding [x1, y1, x2, y2] trên thang nguyên 0..1000.
    """
    xc, yc, w, h = yolo
    x1 = round((xc - w / 2.0) * 1000.0)
    y1 = round((yc - h / 2.0) * 1000.0)
    x2 = round((xc + w / 2.0) * 1000.0)
    y2 = round((yc + h / 2.0) * 1000.0)

    # Kẹp giá trị (clamp) vào đoạn [0, 1000] để chống tràn viền
    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))

    return {"label": label, "bbox_2d": [x1, y1, x2, y2]}


def objects_to_answer(objects: list[dict]) -> str:
    """
    Chuyển danh sách objects thành chuỗi JSON chuẩn.
    Sắp xếp các box theo thứ tự không gian: từ trên xuống dưới (y1), từ trái sang phải (x1).
    Điều này giúp quá trình học ngôn ngữ của LLM ổn định và hội tụ nhanh hơn rất nhiều.
    """
    boxes = [yolo_to_box(obj["label"], obj["yolo"]) for obj in objects]
    boxes.sort(key=lambda b: (b["bbox_2d"][1], b["bbox_2d"][0]))
    return json.dumps(boxes, separators=(",", ":"), ensure_ascii=False)

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9